# Volume 6 Drills — Model-Based RL & Planning

Work through each drill problem. Code cells are provided where applicable.

## Drill 1 — Model-Based vs Model-Free

**Question:** What is the key difference between **model-based** and **model-free** RL?

**Answer (fill in):** ___

<details><summary>Answer</summary>

| | Model-Free | Model-Based |
|---|---|---|
| World model | Not learned | Learned: p(s'|s,a) and r(s,a) |
| Planning | No | Yes — can simulate rollouts |
| Sample efficiency | Low | High (can plan with model) |
| Asymptotic performance | Often higher | Can be limited by model error |
| Examples | DQN, PPO, SAC | Dyna-Q, MBPO, AlphaZero, Dreamer |

Model-based RL uses a learned (or given) transition model to **plan** or **generate synthetic experience**, reducing the need for real environment interactions.
</details>

## Drill 2 — MCTS: Four Phases

**Question:** Monte Carlo Tree Search (MCTS) operates in 4 phases. Name them and briefly describe each.

**Answer (fill in):** ___

<details><summary>Answer</summary>

1. **Selection:** Starting from the root, traverse the tree using a selection policy (e.g., UCB1) that balances exploitation of high-value nodes with exploration of less-visited nodes. Descend until a node with unexpanded children is reached.

2. **Expansion:** Add one or more child nodes to the selected leaf node (expand a new action/state).

3. **Simulation (Rollout):** From the newly expanded node, run a **random (or heuristic) rollout** to the end of the game/episode to obtain a value estimate.

4. **Backpropagation:** Propagate the simulation result back up the tree, updating visit counts N(s) and value estimates Q(s,a) for all ancestor nodes.

AlphaZero replaces random rollouts (step 3) with a **neural network value estimate**, which is far more accurate.
</details>

## Drill 3 — Dyna-Q: Why Simulated Steps?

**Question:** In Dyna-Q, after each **real** environment step, the agent performs **k simulated steps** from the learned model. Why is this beneficial?

**Answer (fill in):** ___

<details><summary>Answer</summary>

Real environment interactions are expensive (time, compute, or physical cost). Dyna-Q's simulated steps allow:

1. **Increased update frequency:** For each real transition, k additional Q-value updates are performed, accelerating learning without extra real-world cost.

2. **Better sample efficiency:** The model enables "mental rehearsal" — the agent can practice from previously visited (s, a) pairs even when the environment is not available.

3. **Value propagation:** Simulated rollouts propagate value information across the state space faster than real experience alone.

As k → ∞ and the model is perfect, Dyna-Q approaches dynamic programming (full planning). In practice, even k=5–50 simulated steps per real step dramatically improves sample efficiency.
</details>

## Drill 4 — Code: Tabular Model for Planning

In [ ]:
import numpy as np
from collections import defaultdict

class TabularModel:
    """
    Deterministic tabular model: stores (s, a) -> (r, s') transitions.
    Used in Dyna-Q for planning.
    """

    def __init__(self):
        # model[(s, a)] = (reward, next_state)
        self.model = {}
        self.visited = set()  # track (s, a) pairs we've seen

    def update(self, s, a, r, s_next):
        """Store transition."""
        self.model[(s, a)] = (r, s_next)
        self.visited.add((s, a))

    def sample(self):
        """Sample a random previously-seen (s, a) pair."""
        import random
        s, a = random.choice(list(self.visited))
        r, s_next = self.model[(s, a)]
        return s, a, r, s_next

    def predict(self, s, a):
        """Predict next state and reward for (s, a)."""
        return self.model.get((s, a), None)

# Simulate Dyna-Q planning steps
Q = defaultdict(lambda: np.zeros(4))  # 4-action Q-table
model = TabularModel()
gamma = 0.99
alpha = 0.1

# Simulate some real transitions
real_transitions = [
    (0, 1, 0.0, 1),  # s=0, a=1, r=0, s'=1
    (1, 2, 0.0, 2),
    (2, 0, 1.0, 3),  # reaching goal
    (3, 0, 0.0, 3),  # terminal loop
]

for s, a, r, s_next in real_transitions:
    # Real Q-update
    td_target = r + gamma * Q[s_next].max()
    Q[s][a] += alpha * (td_target - Q[s][a])
    model.update(s, a, r, s_next)

print("Q-values after real transitions:")
for s in range(4):
    print(f"  Q[{s}] = {Q[s].round(4)}")

# Dyna-Q: k simulated planning steps
k = 20
for _ in range(k):
    s_sim, a_sim, r_sim, s_next_sim = model.sample()
    td_target = r_sim + gamma * Q[s_next_sim].max()
    Q[s_sim][a_sim] += alpha * (td_target - Q[s_sim][a_sim])

print(f"\nQ-values after {k} Dyna-Q planning steps:")
for s in range(4):
    print(f"  Q[{s}] = {Q[s].round(4)}")

## Drill 5 — AlphaZero: Neural Network Outputs

**Question:** In AlphaZero, what does the neural network predict given a board state s?

**Answer (fill in):** ___

<details><summary>Answer</summary>

The AlphaZero neural network is a **dual-headed** network:

1. **Policy head:** π(a | s) — a probability distribution over legal moves. This is used to guide the **selection** and **expansion** phases of MCTS (prior probabilities).

2. **Value head:** V(s) ∈ [-1, +1] — the predicted outcome (win/lose/draw) from position s for the current player. This **replaces** the random rollout in classic MCTS.

Both heads share a common trunk (residual convolutional network). The combined MCTS+NN search produces improved policy estimates, which are used as training targets in a self-play loop.
</details>

## Drill 6 — World Models: The "Imagination" Step

**Question:** In world model approaches (Dreamer, etc.), what is the role of the **imagination** step?

**Answer (fill in):** ___

<details><summary>Answer</summary>

The imagination step allows the agent to **plan entirely inside the learned model** without interacting with the real environment:

1. The world model encodes the current observation into a **latent state** z.
2. In imagination, the model **rolls out sequences** of (latent state, action, predicted reward, predicted next latent state) using its learned dynamics.
3. The policy and value function are trained **entirely on these imagined trajectories** — no real environment steps needed for policy improvement.
4. Periodically, real experience updates the world model.

This dramatically improves **sample efficiency**: Dreamer can learn policies from just hundreds of episodes on Atari, vs thousands for model-free methods.
</details>

## Drill 7 — Compounding Error in Model-Based RL

**Question:** Why do model-based methods suffer from **compounding error** during long rollouts?

**Answer (fill in):** ___

<details><summary>Answer</summary>

Every learned model has some prediction error ε per step. When rolling out h steps, errors **compound multiplicatively**:

- After 1 step: error ≈ ε
- After h steps: error ≈ ε × h (linear in optimistic case) or ε^h growth

In practice, model errors cause the predicted state distribution to **drift far from the true distribution** as rollout length increases. Actions that look good in the model may be catastrophically bad in reality.

This is why:
- Dyna-Q uses 1-step model predictions only
- MBPO uses short rollouts (1–5 steps)
- Dreamer uses a recurrent model trained to be accurate even for longer horizons
</details>

## Drill 8 — MBPO: Mitigating Compounding Error

**Question:** How does MBPO (Model-Based Policy Optimization) mitigate the compounding error problem?

**Answer (fill in):** ___

<details><summary>Answer</summary>

MBPO uses **short model rollouts** (typically 1–5 steps) starting from **real states** in the replay buffer:

1. Sample real states from the replay buffer.
2. Roll forward **k steps** using the model (k is small — often k=1).
3. Add the generated transitions to a **synthetic replay buffer**.
4. Train an off-policy algorithm (SAC) on a mix of real and synthetic data.

By keeping rollouts short and starting from real states:
- Compounding errors remain small (bounded by ε × k)
- The distribution stays close to the true distribution
- The synthetic data provides a large effective batch size

MBPO achieves SAC-level asymptotic performance with 5-40× fewer environment interactions.
</details>

## Drill 9 — When to Prefer Model-Based RL

**Question:** In what situations is model-based RL preferred over model-free RL?

**Answer (fill in):** ___

<details><summary>Answer</summary>

Model-based RL is preferred when:

1. **Environment is expensive to query** — real robots, physical experiments, medical trials. Each interaction costs time, money, or risk.

2. **Environment is well-structured** — physics simulations, board games, or systems where a model can be accurately learned or is available analytically.

3. **Exploration requires planning** — sparse reward environments where random exploration rarely finds rewards. A model enables goal-directed search.

4. **Transfer and generalization** — a model can be used in new tasks (zero-shot planning) without retraining the policy.

5. **Interpretability** — model rollouts are inspectable, unlike black-box model-free policies.

Model-free is preferred when: environment is cheap, high-dimensional observations (raw pixels) are hard to model, or asymptotic performance matters more than sample efficiency.
</details>

## Drill 10 — Code: 1-Step Lookahead Value

In [ ]:
import numpy as np

def one_step_lookahead(state, V, model, actions, gamma=0.99):
    """
    Compute Q(s, a) for all actions using 1-step lookahead.

    model(s, a) -> (reward, next_state)  [deterministic model]
    V[s] -> estimated value of state s
    Returns: array of Q(s, a) for each action.
    """
    Q_values = np.zeros(len(actions))
    for i, a in enumerate(actions):
        result = model(state, a)
        if result is not None:
            reward, next_state = result
            Q_values[i] = reward + gamma * V.get(next_state, 0.0)
        else:
            Q_values[i] = V.get(state, 0.0)  # no transition known
    return Q_values

# Example: 4-state chain: 0 -> 1 -> 2 -> 3 (goal)
def chain_model(s, a):
    """Simple deterministic chain world. Action 0 = move right."""
    if a == 0 and s < 3:
        reward = 1.0 if s == 2 else 0.0
        return reward, s + 1
    return 0.0, s  # stay

# Value estimates from some policy
V = {0: 0.0, 1: 0.5, 2: 0.99, 3: 1.0}
actions = [0, 1]  # 0=right, 1=stay

for state in range(4):
    Q = one_step_lookahead(state, V, chain_model, actions)
    best_action = np.argmax(Q)
    print(f"s={state}: Q={Q.round(4)}  best_action={best_action}")